# Impute

In [ ]:
from ATLAS.Utils.analysisu import *
data_path = '/scratchdata1/MouseBrainAtlases/'
animal = 'WTM01'
section = 'WTM01_7.8'
adata = anndata.read_h5ad(os.path.join(data_path,animal,'Layer','cell_layer.h5ad'))
adata = adata[adata.obs['Slice']==section].copy()
adata

In [ ]:
imputation_cells = np.unique(adata.obs[[i for i in adata.obs.columns if 'neighbor' in i]].values)
imputation_cells = np.unique([i.split('raise')[0] for i in imputation_cells])
imputation_cells.shape

In [ ]:
import gc
imputation_reference_adata = []
# V3
data_path = '/orangedata/ExternalData/Allen_WMB_2024Mar06/expression_matrices/WMB-10Xv3/20230630'
for file in tqdm(os.listdir(data_path)):
    if 'raw.h5ad' in file:
        temp_adata = anndata.read_h5ad(os.path.join(data_path,file))
        shared_cells = np.intersect1d(imputation_cells,temp_adata.obs_names)
        imputation_reference_adata.append(temp_adata[shared_cells].copy())
        del temp_adata
        gc.collect()

#V2
data_path = '//orangedata/ExternalData/Allen_WMB_2024Mar06/expression_matrices/WMB-10Xv2/20230630/'
for file in tqdm(os.listdir(data_path)):
    if 'raw.h5ad' in file:
        temp_adata = anndata.read_h5ad(os.path.join(data_path,file))
        shared_cells = np.intersect1d(imputation_cells,temp_adata.obs_names)
        imputation_reference_adata.append(temp_adata[shared_cells].copy())
        del temp_adata
        gc.collect()
imputation_reference_adata = anndata.concat(imputation_reference_adata)
gc.collect()
imputation_reference_adata.write('/scratchdata1/MouseBrainAtlases/AnalysisNotebooks/WTM01_7.8_imputation_reference.h5ad')


In [ ]:
neighbor_columns = [i for i in adata.obs.columns if 'neighbor' in i]
imputed = None
for i,column in tqdm(enumerate(neighbor_columns)):
    neighbors = [i.split('raise')[0] for i in adata.obs[column]]
    neighbor = imputation_reference_adata[neighbors,:].X.toarray().astype(float)
    if isinstance(imputed,type(None)):
        imputed = neighbor
    else:
        imputed = imputed + neighbor
imputed = imputed/len(neighbor_columns)
imputed_adata = anndata.AnnData(X=imputed,obs=adata.obs,var=imputation_reference_adata.var)
imputed_adata.write('/scratchdata1/MouseBrainAtlases/AnalysisNotebooks/WTM01_7.8_imputation.h5ad')

# Gene Validation

In [ ]:
from ATLAS.Utils.analysisu import *

import pandas as pd
import anndata as anndata
import numpy as np
import itertools

from scipy.spatial import cKDTree
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import min_weight_full_bipartite_matching
from scipy.optimize import linear_sum_assignment
from scipy.sparse import issparse
import matplotlib.colors as mcolors
import scanpy as sc

In [2]:
def greedy_correlation_matching_of_proximal_cells(adata1_org, adata2_org, dist_thresh = 0.1,log=False):
    
    # make copies as we are going to subsample
    adata1 = adata1_org.copy()
    adata2 = adata2_org.copy()

    if issparse(adata1.X):
        adata1.X = adata1.X.toarray()
    if issparse(adata2.X):
        adata2.X = adata2.X.toarray()

    y_min = np.max([np.min(adata1.obs['aligned_ccf_y']),np.min(adata2.obs['aligned_ccf_y'])])
    y_max = np.min([np.max(adata1.obs['aligned_ccf_y']),np.max(adata2.obs['aligned_ccf_y'])])
    z_min = np.max([np.min(adata1.obs['aligned_ccf_z']),np.min(adata2.obs['aligned_ccf_z'])])
    z_max = np.min([np.max(adata1.obs['aligned_ccf_z']),np.max(adata2.obs['aligned_ccf_z'])])
    m = (adata1.obs['aligned_ccf_y']>y_min) & (adata1.obs['aligned_ccf_y']<y_max) & (adata1.obs['aligned_ccf_z']>z_min) & (adata1.obs['aligned_ccf_z']<z_max)
    adata1 = adata1[m].copy()
    m = (adata2.obs['aligned_ccf_y']>y_min) & (adata2.obs['aligned_ccf_y']<y_max) & (adata2.obs['aligned_ccf_z']>z_min) & (adata2.obs['aligned_ccf_z']<z_max)
    adata2 = adata2[m].copy()

    # ensure var paired
    shared_var = adata1.var_names.intersection(adata2.var_names)
    adata1 = adata1[:,shared_var].copy()
    adata2 = adata2[:,shared_var].copy()

    print(adata1.shape,adata2.shape)

    # Remove cells with no expression
    adata1 = adata1[adata1.X.sum(axis=1)>0,:]
    adata2 = adata2[adata2.X.sum(axis=1)>0,:]

    print(adata1.shape,adata2.shape)

     # subsample to the same size
    if adata1.shape[0]<adata2.shape[0]:
        random_indexes = np.random.choice(adata2.obs_names, size=adata1.shape[0], replace=False)
        adata2 = adata2[random_indexes]
    elif adata1.shape[0]>adata2.shape[0]:
        random_indexes = np.random.choice(adata1.obs_names, size=adata2.shape[0], replace=False)
        adata1 = adata1[random_indexes]

    G1 = adata1.X.copy()
    G2 = adata2.X.copy()

    # check for sparse matrix
    if issparse(G1):
        G1 = G1.toarray()
    if issparse(G2):
        G2 = G2.toarray()

    G1 = np.array(G1)
    G2  = np.array(G2)
    if log:
        # if np.nanmin(G1)>0:
        G1 = np.log10(G1+1)
        # if np.nanmin(G2)>0:
        G2 = np.log10(G2+1)

    # create pairing between all adata1 / adata2 points that are withint dist_thresh
    XY1 = np.array(adata1.obs[['aligned_ccf_y','aligned_ccf_z']])
    XY2 = np.array(adata2.obs[['aligned_ccf_y','aligned_ccf_z']])

    XY1 = torch.tensor(XY1)
    XY2 = torch.tensor(XY2)
    distances = torch.cdist(XY1, XY2)
    cells_ix_1,cells_ix_2 = np.where(distances < dist_thresh)
    cells_ix_1_obs_labels = adata1.obs.index[cells_ix_1]
    cells_ix_2_obs_labels = adata2.obs.index[cells_ix_2]
    del distances
    # cells_ix_1 = cells_ix_1.numpy()
    # cells_ix_2 = cells_ix_2.numpy()

    
    # Extract the relevant subsets
    G1_subset = G1[cells_ix_1,:]
    G2_subset = G2[cells_ix_2,:]

    # Calculate means
    G1_mean = np.mean(G1_subset, axis=1)[:,np.newaxis]
    G2_mean = np.mean(G2_subset, axis=1)[:,np.newaxis]

    # Calculate correlation
    numerator = np.sum((G1_subset - G1_mean) * (G2_subset - G2_mean), axis=1)
    denominator = np.sqrt(np.sum((G1_subset - G1_mean)**2, axis=1) * np.sum((G2_subset - G2_mean)**2, axis=1))

    # Handle potential division by zero
    correlations = np.where(denominator != 0, numerator / denominator, 0)


    # Sort correlations in descending order
    sorted_indices = np.argsort(-correlations)
    
    # Initialize sets to keep track of matched rows and columns
    matched_1 = set()
    matched_2 = set()
    good_idxes = []
    for idx in tqdm(sorted_indices):
        c1 = cells_ix_1[idx]
        c2 = cells_ix_2[idx]
        
        # If both row and column are unmatched, add to matches
        if c1 not in matched_1: 
            if  c2 not in matched_2:
                matched_1.add(c1)
                matched_2.add(c2)
                good_idxes.append(idx)
    good_idxes = np.array(good_idxes)
    matched_1 = np.array(cells_ix_1[good_idxes])
    matched_2 = np.array(cells_ix_2[good_idxes])
    cell_correlations = np.array(correlations[good_idxes])

    if issparse(adata1.X):
        paired_adata = anndata.AnnData(X=adata1.X.toarray()[matched_1,:].copy(),var=adata1.var)
        paired_adata.layers['G1'] = np.array(adata1.X.toarray()[matched_1,:].copy())
    else:
        paired_adata = anndata.AnnData(X=adata1.X[matched_1,:].copy(),var=adata1.var)
        paired_adata.layers['G1'] = np.array(adata1.X[matched_1,:].copy())
    if issparse(adata2.X):
        paired_adata.layers['G2'] = np.array(adata2.X.toarray()[matched_2,:].copy())
    else:
        paired_adata.layers['G2'] = np.array(adata2.X[matched_2,:].copy())
    paired_adata.var['G1'] = adata1.var_names
    paired_adata.var['G2'] = adata2.var_names
    paired_adata.obs['G1_cells'] = np.array(adata1.obs.index)[matched_1]
    paired_adata.obs['G2_cells'] = np.array(adata2.obs.index)[matched_2]
    paired_adata.obs['G1_aligned_ccf_y'] = adata1.obs['aligned_ccf_y'].values[matched_1]
    paired_adata.obs['G1_aligned_ccf_z'] = adata1.obs['aligned_ccf_z'].values[matched_1]
    paired_adata.obs['G2_aligned_ccf_y'] = adata2.obs['aligned_ccf_y'].values[matched_2]
    paired_adata.obs['G2_aligned_ccf_z'] = adata2.obs['aligned_ccf_z'].values[matched_2]

    print(paired_adata)

    # Extract the relevant subsets
    G1_subset = np.array(paired_adata.layers['G1']).copy().T#G1_matched.numpy().T
    G2_subset = np.array(paired_adata.layers['G2']).copy().T#G2_matched.numpy().T

    if log:
        G1_subset = np.log10(G1_subset + 1)
        G2_subset = np.log10(G2_subset + 1)
    # Calculate means
    G1_mean = np.mean(G1_subset, axis=1)[:,np.newaxis]
    G2_mean = np.mean(G2_subset, axis=1)[:,np.newaxis]

    # Calculate correlation
    numerator = np.sum((G1_subset - G1_mean) * (G2_subset - G2_mean), axis=1)
    denominator = np.sqrt(np.sum((G1_subset - G1_mean)**2, axis=1) * np.sum((G2_subset - G2_mean)**2, axis=1))

    # Handle potential division by zero
    gene_correlations = np.where(denominator != 0, numerator / denominator, 0)

    cell_correlations = np.array(cell_correlations)#.numpy()
    gene_correlations = np.array(gene_correlations)#.numpy()
    gene_mu = np.mean(paired_adata.layers['G1'],axis=0)
    genes = adata1.var_names

    paired_adata.obs['correlation'] = cell_correlations
    paired_adata.var['correlation'] = gene_correlations
    
    return cell_correlations, gene_correlations, gene_mu, genes,paired_adata

In [ ]:
# Load Datasets 
notebook_path = '/scratchdata1/MouseBrainAtlases/AnalysisNotebooks/'

datasets = [
    'Allen_7.8.h5ad', 
    'Zhuang_Imputed_7.8.h5ad',
    'Zhuang_7.8.h5ad',
    'StarMAP_Imputed_7.8.h5ad',
    'WTM01_7.8_imputation.h5ad']
    
    data = {}
import mygene
mg = mygene.MyGeneInfo()
common_genes = None
for dataset in datasets:
    print(dataset)
    data[dataset] = anndata.read_h5ad(os.path.join(notebook_path,dataset), backed='r')
    if 'ENS' in data[dataset].var.index[0]:
        gene_info = mg.querymany(np.array(data[dataset].var.index), scopes='ensembl.gene', fields='symbol', species='mouse', verbose=False)
        gene_names = {item['query']: item.get('symbol', 'N/A') for item in gene_info}
        data[dataset].var['updated_gene_names'] = [gene_names[i].upper() for i in data[dataset].var.index]
    else:
        data[dataset].var['updated_gene_names'] = [i.upper() for i in data[dataset].var.index]
    genes = set(data[dataset].var['updated_gene_names'].values)
    if common_genes is None:
        common_genes = genes
    else:
        common_genes &= genes
    print(len(common_genes))
common_genes

In [ ]:
# Filter to common gene set and save
common_gene_data = {}
for dataset in datasets:
    common_gene_data[dataset] = data[dataset][:,data[dataset].var['updated_gene_names'].isin(common_genes)].to_memory().copy()
    if not 'aligned_ccf_y' in common_gene_data[dataset] .obs.columns:
        common_gene_data[dataset].obs['aligned_ccf_y'] = common_gene_data[dataset].obs['ccf_y']
        common_gene_data[dataset].obs['aligned_ccf_x'] = common_gene_data[dataset].obs['ccf_x']
        common_gene_data[dataset].obs['aligned_ccf_z'] = common_gene_data[dataset].obs['ccf_z']
    common_gene_data[dataset].write(os.path.join(notebook_path,dataset.replace('.h5ad','_common_genes.h5ad')))
    print(dataset)
    print(common_gene_data[dataset])

In [4]:
# Load from file
common_gene_data = {}
for dataset in datasets:
    common_gene_data[dataset] = anndata.read_h5ad(os.path.join(notebook_path,dataset.replace('.h5ad','_common_genes.h5ad')))

In [ ]:
# visual inspection
for dataset in datasets:
    adata = common_gene_data[dataset]
    plt.figure()
    plt.title(dataset)
    plt.scatter(adata.obs['aligned_ccf_z'],adata.obs['aligned_ccf_y'],s=0.1)
    plt.show()

In [ ]:
# Create pairwise events
combinations = list(itertools.combinations(np.arange(len(datasets)), 2))
""" Pairwise correlation analysis """
for i,comb in enumerate(combinations):
    dataset1 = common_gene_data[datasets[comb[0]]].copy()
    dataset2 = common_gene_data[datasets[comb[1]]].copy()
    dataset1.var.index = dataset1.var['updated_gene_names'].values
    dataset2.var.index = dataset2.var['updated_gene_names'].values
    d = datasets[comb[0]]+datasets[comb[1]]
    if ('Zhuang' in d) & ('StarMAP' in d):
        dataset1.obs['aligned_ccf_z'] = np.abs(dataset1.obs['aligned_ccf_z']-5.71)
        dataset2.obs['aligned_ccf_z'] = np.abs(dataset2.obs['aligned_ccf_z']-5.71)
    print(datasets[comb[0]],datasets[comb[1]])
    cell_correlations, gene_correlations, gene_mu, genes,paired_adata = greedy_correlation_matching_of_proximal_cells(dataset1,dataset2, dist_thresh = 0.1,log=True)    
    paired_adata.write(f"/scratchdata1/MouseBrainAtlases/AnalysisNotebooks/{datasets[comb[0]].split('.')[0]}_{datasets[comb[1]].split('.')[0]}_paired.h5ad")
    del paired_adata
    gc.collect()


In [ ]:
# Generate Key for plotting
all_datasets = datasets
avg_corr_log = np.ones((len(all_datasets),len(all_datasets),3))
for i,comb in enumerate(combinations):
    paired_adata = anndata.read_h5ad(f"/scratchdata1/MouseBrainAtlases/AnalysisNotebooks/{datasets[comb[0]].split('.')[0]}_{datasets[comb[1]].split('.')[0]}_paired.h5ad")
    # avg_corr_log[comb[0],comb[1],:] = plt.cm.tab10.colors[i]
    avg_corr_log[comb[1],comb[0],:] = plt.cm.tab10.colors[i]

text_converter = {d:d.split('_7')[0].replace('_',' ') for d in datasets}#{'Val': 'MERFISH', 'Z': 'Zhuang', 'Z-Imp': 'Zhuang Imputed', 'A': 'Allen', 'Imp': 'ATLAS Imputed'}
text_converter['WTM01_7.8_imputation.h5ad'] = ('ATLAS_Imputed').replace('_',' ')
for key,label in text_converter.items():
    if 'Imputed' in label:
        continue
    else:
        text_converter[key] = label+' MERFISH'
# text_converter = {'Allen_7.8.h5ad': 'MERSCOPE \n Measured \n (MSM)',
#  'Zhuang_Imputed_7.8.h5ad': 'MERFISH \n Imputed \n (MFI)',
#  'Zhuang_7.8.h5ad': 'MERFISH \n Measured \n (MFM)',
#  'StarMAP_Imputed_7.8.h5ad': 'STARmap \n Imputed \n (SMI)',
#  'WTM01_7.8_imputation.h5ad': 'ATLAS \n Imputed \n (AI)'}
text_converter = {'Allen_7.8.h5ad': 'MSM',
 'Zhuang_Imputed_7.8.h5ad': 'MFI',
 'Zhuang_7.8.h5ad': 'MFM',
 'StarMAP_Imputed_7.8.h5ad': 'SMI',
 'WTM01_7.8_imputation.h5ad': 'AI'}
fig, ax1 = plt.subplots(1, 1, figsize=(5, 5))
temp = avg_corr_log.copy()
# temp[temp==0] = np.nan
im2 = ax1.imshow(temp)#,cmap='jet',vmin=0,vmax=1)
# ax1.set_xticks(range(len(all_datasets)))
# ax1.set_xticklabels([text_converter[i] for i in all_datasets], rotation=45, ha='right', color='black')
# ax1.set_yticks(range(len(all_datasets)))
# ax1.set_yticklabels([text_converter[i] for i in all_datasets], color='black')
# fig.colorbar(im2, ax=ax1, label='Average Correlation')
plt.grid(False)
# plt.axis('off')
# plt.tight_layout()
ax1.spines['top'].set_color('white')
ax1.spines['bottom'].set_color('white')
ax1.spines['left'].set_color('white')
ax1.spines['right'].set_color('white')
ax1.tick_params(axis='x', colors='white')
ax1.tick_params(axis='y', colors='white')

ax1.set_xticks(range(len(all_datasets)))
ax1.set_xticklabels([text_converter[i] for i in all_datasets], color='black',rotation=-90)
ax1.set_yticks(range(len(all_datasets)))
ax1.set_yticklabels([text_converter[i] for i in all_datasets],ha='center',color='black',rotation=-90)
ax1.tick_params(axis='y', pad=10)
plt.savefig(os.path.join(notebook_path,'pairwise_color_decoder_borrom_right.svg'),dpi=300)
plt.show()

In [7]:
text_converter = {'Allen_7.8.h5ad': 'MSM',
 'Zhuang_Imputed_7.8.h5ad': 'MFI',
 'Zhuang_7.8.h5ad': 'MFM',
 'StarMAP_Imputed_7.8.h5ad': 'SMI',
 'WTM01_7.8_imputation.h5ad': 'AI'}

In [ ]:
# Visual Inspection: ROC Curves 
plt.figure(figsize=[7, 5])
order = np.array(range(len(combinations)))
text_converter = {'Allen_7.8.h5ad': 'MSM',
 'Zhuang_Imputed_7.8.h5ad': 'MFI',
 'Zhuang_7.8.h5ad': 'MFM',
 'StarMAP_Imputed_7.8.h5ad': 'SMI',
 'WTM01_7.8_imputation.h5ad': 'AI'}

handles_labels = []

for i in order:
    comb = combinations[i]
    d1 = datasets[comb[0]]
    d2 = datasets[comb[1]]
    paired_adata = anndata.read_h5ad(f"/scratchdata1/MouseBrainAtlases/AnalysisNotebooks/{datasets[comb[0]].split('.')[0]}_{datasets[comb[1]].split('.')[0]}_paired.h5ad")
    correlations = paired_adata.obs['correlation'].values
    thresholds = np.arange(-0.3,1.1,0.1)
    correlations = correlations[~np.isnan(correlations)]
    y = np.arange(correlations.shape[0]) / correlations.shape[0]
    x = np.sort(correlations)[::-1]
    if ('ATLAS' in d1)| ('ATLAS' in d2):
        line, = plt.plot(x, y, label=f"{text_converter[d1]} vs. {text_converter[d2]}",color=plt.cm.tab10.colors[i])
    else:
        line, = plt.plot(x, y, label=f"{text_converter[d1]} vs. {text_converter[d2]}",color=plt.cm.tab10.colors[i])
    
plt.grid(False)
plt.xlabel('Correlation Cutoff')
plt.ylabel('Fraction of Cells')
plt.xlim((1,-0.3))
plt.ylim((0,1))
plt.savefig(os.path.join(notebook_path, 'cell_correlation_fraction_ROC.svg'), dpi=300)
plt.show()

In [ ]:
# Visual Inspection: Gene Calls

text_converter = {'Allen_7.8.h5ad': 'MSM',
 'Zhuang_Imputed_7.8.h5ad': 'MFI',
 'Zhuang_7.8.h5ad': 'MFM',
 'StarMAP_Imputed_7.8.h5ad': 'SMI',
 'WTM01_7.8_imputation.h5ad': 'AI'}


ordered_datasets = ['WTM01_7.8_imputation.h5ad','Zhuang_7.8.h5ad','Zhuang_Imputed_7.8.h5ad','Allen_7.8.h5ad','StarMAP_Imputed_7.8.h5ad']
highest_correlation = gene_correlations_out.mean(1).sort_values(ascending=False)
for g,gene in tqdm(enumerate(highest_correlation.index.values)):
    score = highest_correlation[g]
    fig,axs = plt.subplots(1,len(datasets),figsize=(len(datasets)*3.5,5))
    axs = axs.ravel()
    for i,dataset in enumerate(ordered_datasets):
        adata = common_gene_data[dataset].copy()
        if text_converter[dataset]=='MSM':
            adata = adata[adata.obs['aligned_ccf_z']<5.71]
        if text_converter[dataset]=='AI':
            adata = adata[adata.obs['aligned_ccf_z']<5.71]
        if text_converter[dataset]=='SMI':
            adata.obs['aligned_ccf_z'] = np.abs(-1*(adata.obs['aligned_ccf_z']-5.17)+5.71)

        gidx = np.where(adata.var['updated_gene_names'].values == gene)[0][0]
        if issparse(adata.X):
            c=adata.X[:, gidx].toarray()
        else:
            c=adata.X[:, gidx]
        c = c.ravel()
        vmin,vmax = np.percentile(c[(np.isnan(c)==False)], [5, 90])
        ax = axs[i]
        order = np.argsort(c)
        ax.scatter(adata.obs['aligned_ccf_z'][order], adata.obs['aligned_ccf_y'][order], s=100, marker='x',c='r')
        ax.scatter(adata.obs['aligned_ccf_z'][order], adata.obs['aligned_ccf_y'][order], s=50, marker='x',c='w')
        ax.scatter(adata.obs['aligned_ccf_z'][order], adata.obs['aligned_ccf_y'][order], s=0.1, marker='x',c=c[order], label=dataset,vmin=vmin,vmax=vmax,cmap='Grays')
        ax.set_ylim([8,0])
        ax.axis('off')
    plt.subplots_adjust(left=0.05, right=0.95, top=0.95, bottom=0.05, wspace=0.1, hspace=0.1)
    fig.text(-0.01, 0.5, gene, ha='center', va='center', rotation=90, fontsize=75)
    plt.tight_layout()
    plt.savefig(os.path.join(notebook_path, f'Gray_with_border_gene_expression_{gene}_{round(score,2)}.png'), dpi=300)
    if g<10:
        plt.show()
    else:
        plt.close()

# Cell Type Validation

In [13]:
reference_data = common_gene_data['Allen_7.8.h5ad']
measured_data = common_gene_data['WTM01_7.8_imputation.h5ad']

In [ ]:
# Qualitative Visual inspection of locations
ordered_datasets = ['WTM01_7.8_imputation.h5ad','Allen_7.8.h5ad']

fig,axs = plt.subplots(1,len(ordered_datasets),figsize=(len(ordered_datasets)*5,7))
axs = axs.ravel()
for i,dataset in enumerate(ordered_datasets):
    adata = common_gene_data[dataset].copy()
    if text_converter[dataset]=='MSM':
        adata = adata[adata.obs['aligned_ccf_z']<5.71]
    if text_converter[dataset]=='AI':
        adata = adata[adata.obs['aligned_ccf_z']<5.71]
    if text_converter[dataset]=='SMI':
        adata.obs['aligned_ccf_z'] = np.abs(-1*(adata.obs['aligned_ccf_z']-5.17)+5.71)

    ax = axs[i]
    ax.scatter(adata.obs['aligned_ccf_z'], adata.obs['aligned_ccf_y'], s=0.1, c=adata.obs['subclass_color'])
    ax.set_title(text_converter[dataset])
    ax.axis('off')
    ax.axis('equal')
plt.tight_layout()
plt.savefig(os.path.join(notebook_path, f'subclass_locations.png'), dpi=300)


In [ ]:
# Composition Scatter plot

import pandas as pd
import numpy as np
import os
notebook_path = '/scratchdata1/MouseBrainAtlases/AnalysisNotebooks/'


ref_df = pd.DataFrame(reference_data.obs['subclass'].value_counts())
ref_df.columns = ['reference_count']
meas_df = pd.DataFrame(measured_data.obs['subclass'].value_counts())
meas_df.columns = ['measured_count']
combined_df = ref_df.join(meas_df, how='outer').fillna(0)

reference_count = combined_df['reference_count']
measured_count = combined_df['measured_count']

# Calculate correlation in linear scale
linear_corr, linear_p_value = pearsonr(reference_count, measured_count)

# Calculate correlation in log scale
log_reference_count = np.log10(reference_count + 1)  # Adding 1 to avoid log(0)
log_measured_count = np.log10(measured_count + 1)   # Adding 1 to avoid log(0)
log_corr, log_p_value = pearsonr(log_reference_count, log_measured_count)

# Print the correlation results
print(f"Linear scale correlation: {linear_corr:.2f} (p-value: {linear_p_value:.2e})")
print(f"Log scale correlation: {log_corr:.2f} (p-value: {log_p_value:.2e})")

# Plot the scatter plot in log scale
plt.figure(figsize=(4, 4))
converter = dict(zip(allen_adata.obs['subclass'].unique(),allen_adata.obs['subclass_color'].unique()))
plt.subplot(1, 1, 1)
plt.scatter(combined_df['reference_count']+1, combined_df['measured_count']+1,s=3,c=combined_df.index.map(converter))
plt.xscale('log')
plt.yscale('log')
plt.xlabel('MERSCOPE', fontsize=16)
plt.ylabel('ATLAS', fontsize=16)
plt.grid(False)
plt.axis('equal')
plt.tick_params(axis='both', which='major', labelsize=16)
# plt.title('Scatter Plot (Log Scale)')
plt.text(0.05, 0.95, f'{log_corr:.2f}',
         transform=plt.gca().transAxes, fontsize=16, verticalalignment='top')
plt.savefig(os.path.join(notebook_path, 'scatter_plot_comparison_ATLAS_MERSCOPE_Whole.svg'), dpi=300)
plt.tight_layout()
plt.tight_layout()

# Marker Genes

In [34]:
project_path = '/scratchdata1/Images2024/Haley/ATLAS'
analysis_path = '/scratchdata1/MouseBrainAtlases'
animal = 'RNA'
basepath = os.path.join(analysis_path, animal)
wollman_merfish = anndata.read_h5ad('/scratchdata1/RW_matching_unpaired_sections/merfish_validation.h5ad')
wollman_ATLAS = anndata.read_h5ad(os.path.join(basepath,'posterior_adata.h5ad'))
allen_merfish = anndata.read_h5ad('/scratchdata1/RW_matching_unpaired_sections/merfish_Allen.h5ad')


In [41]:
data_dict = {}
data_dict['MERFISH'] = wollman_merfish
data_dict['Imputed'] = wollman_ATLAS
data_dict['Allen'] = allen_merfish

In [ ]:
id_a = np.array(data_dict['MERFISH'].obs['ATLAS_cell_id'])
id_b = np.array(data_dict['Imputed'].obs.index)
# Find the common values
shared_ids = np.intersect1d(id_a, id_b)
adata = data_dict['MERFISH'][data_dict['MERFISH'].obs['ATLAS_cell_id'].isin(shared_ids),:].copy()
adata.obs['atlas_subclass'] = np.array(data_dict['Imputed'].obs.loc[adata.obs['ATLAS_cell_id'],'subclass'])
adata = adata[adata.obs['ATLAS_cell_distance']<4].copy()
top_cts = adata.obs['atlas_subclass'].value_counts().index[:25]
adata = adata[adata.obs['atlas_subclass'].isin(top_cts),:].copy()
converter = dict(zip(data_dict['Allen'].var.index,data_dict['Allen'].var['gene_symbol']))
adata.var.index = pd.Series(adata.var.index).map(converter)
adata.var.index.name=''
X = pd.DataFrame(adata.X, columns=adata.var.index, index=adata.obs.index)
X['subclass'] = adata.obs['atlas_subclass']
import seaborn as sns
X = X.groupby('subclass').mean()
X_copy = X.copy()
X = (X-np.median(X,axis=0))/np.std(X,axis=0)
measured_X = X.copy()
adata = data_dict['Allen'].copy()
converter = dict(zip(data_dict['Allen'].var.index,data_dict['Allen'].var['gene_symbol']))
adata.var.index = pd.Series(adata.var.index).map(converter)
adata.var.index.name=''
# print(adata)
X = pd.DataFrame(adata.X.toarray(), columns=adata.var.index, index=adata.obs.index)
X['subclass'] = adata.obs['subclass']
import seaborn as sns
X = X.groupby('subclass').mean()
X = (X-np.median(X,axis=0))/np.std(X,axis=0)
reference_X = X.copy()

shared_cts = measured_X.index.intersection(reference_X.index)
shared_genes = measured_X.columns.intersection(reference_X.columns)

reference_X = reference_X.loc[shared_cts,shared_genes]
measured_X = measured_X.loc[shared_cts,shared_genes]

In [ ]:
import numpy as np
from scipy.stats import pearsonr
import seaborn as sns
import matplotlib.pyplot as plt


# Calculate correlation for each column
column_correlations = {}
for col in reference_X.columns:
    corr, _ = pearsonr(reference_X[col], measured_X[col])
    column_correlations[col] = corr

# Sort columns by correlation and select top 25
sorted_columns = sorted(column_correlations, key=column_correlations.get, reverse=True)[:25]

# Calculate correlation for each row
row_correlations = {}
for idx in reference_X.index:
    corr, _ = pearsonr(reference_X.loc[idx], measured_X.loc[idx])
    row_correlations[idx] = corr

# Sort rows by correlation and select top 25
sorted_rows = sorted(row_correlations, key=row_correlations.get, reverse=True)[:25]

# Select the top 25 columns and rows
top_reference_X = reference_X.loc[sorted_rows, sorted_columns]
top_measured_X = measured_X.loc[sorted_rows, sorted_columns]

# Create a clustermap for top_reference_X
clustermap = sns.clustermap(top_reference_X, cmap='coolwarm', vmin=-2, vmax=3)

# Extract the row and column order
row_order = clustermap.dendrogram_row.reordered_ind
col_order = clustermap.dendrogram_col.reordered_ind

# Reorder top_measured_X based on the extracted order
top_measured_X_reordered = top_measured_X.iloc[row_order, col_order]

# Plot the heatmaps side by side
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
# Process the reference data
r = top_reference_X.iloc[row_order, col_order]
r.index = [i.split(' ')[0] for i in r.index]
sns.heatmap(r, ax=axes[0], cmap='coolwarm', vmin=-2, vmax=3, cbar=False, xticklabels=True)
axes[0].set_title('MERSCOPE')
# axes[0].set_xlabel('Columns')
axes[0].set_ylabel('')

# Process the measured data
m = top_measured_X_reordered
m.index = [i.split(' ')[0] for i in m.index]
sns.heatmap(m, ax=axes[1], cmap='coolwarm', vmin=-2, vmax=3, cbar=False, xticklabels=True)
axes[1].set_title('ATLAS + MERFISH')
# axes[1].set_xlabel('Columns')
axes[1].set_ylabel('')

# Adjust x-axis labels
plt.setp(axes[0].get_xticklabels(), rotation=45, ha='right')
plt.setp(axes[1].get_xticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.savefig(os.path.join(notebook_path, 'marker_genes_MERSCOPE_ATLAS.svg'), dpi=300)
plt.show()